# 3 · Sensitivity to conductivity, kinetics and agglomerate transport

The public defaults are generic (docs/parameters.md). This notebook shows how the rate capability depends on three of them:

- the electronic conductivity σ;
- the rate constant k;
- transport inside the agglomerates, varied through the tortuosity of their pores.

Each case is a discharge to 3.0 V at 1C, 3C and 10C on a coarse grid. With the defaults the cell is not strongly rate-limited below about 2C, so the ranges below reach into the limiting regimes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace

from nmc_model.agglomerate import AggParams
from nmc_model.agglomerate_corrected import run_corrected

base = AggParams(mode="corrected", nj=30, nja=10, dt_s=10.0)
rates = [1.0, 3.0, 10.0]

def capacity(p):
    r = run_corrected(p)
    return r.array[-1, 2] * 96485.0 / (p.M * 3.6)      # mAh/g at the cutoff

def sweep(name, values, label):
    for v in values:
        caps = [capacity(replace(base, C_rate=c, **{name: v})) for c in rates]
        plt.plot(rates, caps, "o-", label=f"{label} = {v:g}")
    plt.xlabel("C-rate"); plt.ylabel("capacity to 3.0 V [mAh/g]"); plt.legend(); plt.show()

## Electronic conductivity

In [ ]:
sweep("sigma", [0.001, 0.01, 0.1], "σ [S/cm]")

## Rate constant

In [ ]:
sweep("rxn_k", [2.5e-8, 2.5e-7, 2.5e-6], "k")

## Transport inside the agglomerates

The default pore tortuosity is Bruggeman's, ε_agg^−½ ≈ 2.2. Larger values slow salt transport inside the agglomerates; around 1000 times Bruggeman's value the agglomerate pores become the bottleneck, their cores run out of salt, and the discharge ends early. The model handles the exhausted cores smoothly (docs/model.md section 8).

In [ ]:
sweep("tau_agg", [2.24, 224.0, 2240.0, 22400.0], "τ_agg")